# Experimento 3: learning rate y paciencia de Early Stopping en R4
R4 (LSTM de 5 caracteres, `clipnorm=1.0`) con lr de 0,005 o 0,0005 y paciencia de 5 o 10.

**Resultado:** la paciencia no cambia nada (se recupera el mismo mínimo de `val_loss`); el lr sí: 49,20 % con 0,005 y 50,07 % con 0,0005. El notebook principal usa 0,0005 y paciencia 5.

In [2]:
# ==============================================================================
# 1.1 CONFIGURACIÓN DEL ENTORNO
# ==============================================================================
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import json
import time
import urllib.request
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 4)

SEED = 42
SEEDS = [42, 123, 456, 789, 2026]
np.random.seed(SEED)

print(f"TensorFlow {tf.__version__} | Keras {keras.__version__}")

TensorFlow 2.21.0 | Keras 3.15.1


In [3]:
# ==============================================================================
# 1.2 CARGA DEL TEXTO LIMPIO Y DE LA PARTICIÓN DE LA ETAPA 1
# ==============================================================================
BASE_DIR = Path.cwd().parent if (Path.cwd().parent / "data").exists() else Path.cwd()
PROC = BASE_DIR / "data" / "processed"

text = (PROC / "The_Festival_clean.txt").read_text(encoding="utf-8")
splits = json.load(open(PROC / "splits_bloques.json", encoding="utf-8"))

L = splits["L"]
limites = splits["limites_bloques"]
bloques_split = splits["bloques"]
vocabulary = list(splits["vocabulario"])
char2idx = {c: i for i, c in enumerate(vocabulary)}
idx2char = {i: c for c, i in char2idx.items()}
# Muestra el espacio y el salto de línea de forma legible en tablas y gráficos
vis = lambda c: {" ": "<SP>", "\n": "<NL>"}.get(c, c)
V = len(vocabulary)

def ventanas_de_bloques(ids):
    X, y = [], []
    for b in ids:
        ini, fin = limites[b]
        for i in range(ini, fin - L):
            X.append(text[i:i+L])
            y.append(text[i+L])
    return X, y

X_train, y_train = ventanas_de_bloques(bloques_split["train"])
X_dev,   y_dev   = ventanas_de_bloques(bloques_split["dev"])

# Entrada como índices enteros: (N, L)
a_indices = lambda X: np.array([[char2idx[c] for c in x] for x in X], dtype=np.int32)
Xi_train, Xi_dev = a_indices(X_train), a_indices(X_dev)
yi_train = np.array([char2idx[c] for c in y_train], dtype=np.int32)
yi_dev   = np.array([char2idx[c] for c in y_dev], dtype=np.int32)

print(f"|V| = {V} | L = {L}")
print(f"Train: {len(X_train):,} muestras | Dev: {len(X_dev):,} muestras")

|V| = 64 | L = 40
Train: 12,780 muestras | Dev: 2,880 muestras


In [4]:
# ==============================================================================
# 1.3 REFERENCIAS DE LAS ETAPAS 1 Y 2
# ==============================================================================
ref_base = json.load(open(PROC / "baseline_dev.json", encoding="utf-8"))
reg2 = json.load(open(PROC / "resultados_modelos.json", encoding="utf-8"))
ref2 = reg2["configuraciones"][reg2["ganador"]]

# Configuración de entrenamiento heredada del ganador de la Etapa 2
CFG2 = ref2["config"]
LR = CFG2["lr"]
DROPOUT = CFG2.get("dropout", 0.0)
EPOCAS = reg2["learning_rate"]["epocas"]
BATCH = 128
K = ref_base["K"]

print(f"Baseline (Etapa 1): {ref_base['accuracy_dev']*100:.2f}% en Dev")
print(f"Ganador Etapa 2 ({reg2['ganador']}): {ref2['dev_acc_mean']*100:.2f}% ± {ref2['dev_acc_std']*100:.2f}% en Dev")
print(f"Configuración heredada: lr = {LR:g} | Dropout = {DROPOUT} | {EPOCAS} épocas | batch {BATCH} | K = {K}")

Baseline (Etapa 1): 40.83% en Dev
Ganador Etapa 2 (Dropout 0.05): 48.71% ± 0.52% en Dev
Configuración heredada: lr = 0.0005 | Dropout = 0.05 | 100 épocas | batch 128 | K = 5


In [5]:
# ==============================================================================
# 2.1 CONSTRUCCIÓN, ENTRENAMIENTO Y EVALUACIÓN CON 5 SEMILLAS
# ==============================================================================
def construir_modelo(tipo, k=K, emb_dim=16, emb_init=None, emb_trainable=True, dropout=DROPOUT, unidades_lstm=128):
    """
    tipo = "onehot": MLP de la Etapa 2 sobre one-hot aplanado.
    tipo = "densa":  Embedding -> Flatten -> MLP 128-64-32.
    tipo = "lstm":   Embedding -> LSTM -> Dropout -> softmax.
    """
    entrada = layers.Input(shape=(k,), dtype="int32")
    if tipo == "onehot":
        # One-hot exacto: embedding fijo con la matriz identidad
        x = layers.Embedding(V, V, embeddings_initializer=keras.initializers.Identity(), trainable=False)(entrada)
    else:
        init = keras.initializers.Constant(emb_init) if emb_init is not None else "uniform"
        x = layers.Embedding(V, emb_dim, embeddings_initializer=init, trainable=emb_trainable, name="embedding")(entrada)

    if tipo in ("onehot", "densa"):
        x = layers.Flatten()(x)
        for u in [128, 64, 32]:
            x = layers.Dense(u, activation="relu")(x)
            if dropout > 0:
                x = layers.Dropout(dropout)(x)
    else:
        x = layers.LSTM(unidades_lstm)(x)
        if dropout > 0:
            x = layers.Dropout(dropout)(x)

    salida = layers.Dense(V, activation="softmax")(x)
    return keras.Model(entrada, salida)

def evaluar(constructor, k, epochs=EPOCAS, seeds=SEEDS, lr=None, clipnorm=None, use_es=False, patience=5):
    """Entrena con cada semilla y devuelve (resumen, historial y modelo de la semilla 42)."""
    Xtr, Xdv = Xi_train[:, -k:], Xi_dev[:, -k:]
    corridas = []
    for s in seeds:
        keras.utils.set_random_seed(s)
        modelo = constructor()
        modelo.compile(optimizer=keras.optimizers.Adam(learning_rate=lr or LR, clipnorm=clipnorm),
                       loss="sparse_categorical_crossentropy", metrics=["accuracy"])
        cbs = []
        if use_es:
            cbs.append(callbacks.EarlyStopping(monitor="val_loss", patience=patience, restore_best_weights=True))
        t0 = time.perf_counter()
        h = modelo.fit(Xtr, yi_train, validation_data=(Xdv, yi_dev), epochs=epochs, batch_size=BATCH,
                       callbacks=cbs, verbose=0)
        t_run = time.perf_counter() - t0
        if s == SEED:
            h42, m42 = h.history, modelo
        loss_tr, acc_tr = [float(v) for v in modelo.evaluate(Xtr, yi_train, verbose=0)]
        loss_dv, acc_dv = [float(v) for v in modelo.evaluate(Xdv, yi_dev, verbose=0)]
        va = h.history["val_accuracy"]
        corridas.append({"seed": s, "tiempo": t_run, "train_acc": acc_tr, "dev_acc": acc_dv,
                         "brecha": acc_tr - acc_dv, "dev_loss": loss_dv, "dev_perp": float(np.exp(loss_dv)),
                         "pico_acc": float(max(va)), "ep_pico_acc": int(np.argmax(va)) + 1,
                         "epocas": len(h.history["loss"])})
    df_c = pd.DataFrame(corridas)
    resumen = {"params": int(sum(np.prod(w.shape) for w in m42.trainable_weights)), "k": k, "dev_acc_min": float(df_c["dev_acc"].min()), "corridas": corridas}
    for col in df_c.columns.drop("seed"):
        resumen[f"{col}_mean"] = float(df_c[col].mean())
        resumen[f"{col}_std"] = float(df_c[col].std())
    return resumen, h42, m42



In [6]:
import json
cfg_emb = dict(emb_dim=300)
VARIANTES = {
    "R4 actual (lr 5e-3, pac. 5)":   dict(lr=5e-3, patience=5),
    "lr 5e-3, pac. 10":              dict(lr=5e-3, patience=10),
    "lr 5e-4, pac. 5":               dict(lr=5e-4, patience=5),
    "R4 variante (lr 5e-4, pac. 10)": dict(lr=5e-4, patience=10),
}
salida = {}
for n, p in VARIANTES.items():
    r, h, _ = evaluar(lambda: construir_modelo("lstm", k=K, **cfg_emb), k=K, clipnorm=1.0, use_es=True, **p)
    salida[n] = {k: v for k, v in r.items() if k != "corridas"} | {"dev_por_semilla": [x["dev_acc"] for x in r["corridas"]],
                                                                  "ep_por_semilla": [x["epocas"] for x in r["corridas"]]}
    print(f"{n:<32} Dev {r['dev_acc_mean']*100:.2f}% ± {r['dev_acc_std']*100:.2f}% | Train {r['train_acc_mean']*100:.2f}% | "
          f"pico {r['pico_acc_mean']*100:.2f}% (ép. {r['ep_pico_acc_mean']:.0f}) | {r['epocas_mean']:.0f} épocas | {r['tiempo_mean']:.0f} s", flush=True)

R4 actual (lr 5e-3, pac. 5)      Dev 49.20% ± 0.96% | Train 57.81% | pico 49.83% (ép. 9) | 12 épocas | 15 s


lr 5e-3, pac. 10                 Dev 49.20% ± 0.96% | Train 57.81% | pico 49.83% (ép. 9) | 17 épocas | 18 s


lr 5e-4, pac. 5                  Dev 50.07% ± 0.54% | Train 60.50% | pico 50.49% (ép. 43) | 47 épocas | 52 s


R4 variante (lr 5e-4, pac. 10)   Dev 50.07% ± 0.54% | Train 60.50% | pico 50.60% (ép. 46) | 52 épocas | 65 s
